In [13]:
from pathlib import Path
import shutil
import numpy as np
import soundfile as sf
from IPython.display import Audio, display


In [14]:
CLIP_DIR = Path("datasets/ssw_clips")
OUT_DIR = Path("datasets/ssw_clips_mixed")

SNR_DB = [-20, -15, -10, -5, -2.5, 0, 2.5, 5.0]

OUT_DIR.mkdir(parents=True, exist_ok=True)


In [15]:
def rms_power(x: np.ndarray) -> float:
    return float(np.mean(x.astype(np.float32) ** 2))

def mix_at_snr(signal: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    len_signal = len(signal)
    len_noise = len(noise)
    if (len_signal != len_noise):
        print("Signal and noise should be the same length")
    n = min(len(signal), len(noise)) 
    signal, noise = signal[:n], noise[:n]

    power_signal = rms_power(signal)
    power_noise = rms_power(noise)

    if power_signal == 0 or power_noise == 0:
        raise ValueError("Signal or noise is silent")
    
    # SNR_dB = 10log_10(P_signal/P_noise)
    # P_signal/P_noise = 10^(SNR_dB/10)
    # alpha_power = P_signal / P_noise * 1 / 10^(SNR_dB/10)
    # alpha_amplitude = sqrt(alpha_power)

    alpha = np.sqrt(power_signal / (power_noise * 10 ** (snr_db / 10)))
    mixed = signal + alpha * noise
    
    # optional: prevent clipping when writing PCM
    peak = np.max(np.abs(mixed))
    if peak > 1.0:
        mixed = mixed / peak
    return mixed.astype(np.float32)
    

In [16]:
def snr_filename_tag(snr_db: float) -> str:
    """Filename-safe SNR tag, e.g. -20, -2.5, 0, 2.5, 5."""
    return f"{snr_db:g}"


signal_paths = sorted(CLIP_DIR.glob("*_signal.flac"))
print(f"Found {len(signal_paths)} signal clips, SNRs={SNR_DB}")

for sig_path in signal_paths:
    noise_path = CLIP_DIR / sig_path.name.replace("_signal.flac", "_noise.flac")
    if not noise_path.exists():
        print(f"skip {sig_path.name}: missing noise pair")
        continue

    signal, sr_s = sf.read(sig_path, dtype="float32")
    noise, sr_n = sf.read(noise_path, dtype="float32")
    if sr_s != sr_n:
        print(f"skip {sig_path.name}: sr mismatch {sr_s} vs {sr_n}")
        continue

    stem = sig_path.name.replace("_signal.flac", "")  # e.g. SSW_003_..._cangoo_02

    # clean reference copy
    clean_path = OUT_DIR / f"{stem}_clean.flac"
    shutil.copy2(sig_path, clean_path)

    for snr_db in SNR_DB:
        mixed = mix_at_snr(signal, noise, snr_db)
        out_path = OUT_DIR / f"{stem}_{snr_filename_tag(snr_db)}.flac"
        sf.write(out_path, mixed, sr_s)

    print(f"wrote {stem}_clean + {len(SNR_DB)} SNR mixes")

# preview one matched pair family
example = next(OUT_DIR.glob("*_clean.flac"), None)
if example is None:
    print("No outputs yet")
else:
    stem = example.name.replace("_clean.flac", "")
    print("Preview:", stem)
    audio, sr = sf.read(example)
    display(Audio(audio, rate=sr))
    mixed_ex = OUT_DIR / f"{stem}_{snr_filename_tag(SNR_DB[0])}.flac"
    audio, sr = sf.read(mixed_ex)
    display(Audio(audio, rate=sr))


Found 133 signal clips, SNRs=[-20, -15, -10, -5, -2.5, 0, 2.5, 5.0]
wrote SSW_003_20170225_030002Z_cangoo_02_clean + 8 SNR mixes
wrote SSW_003_20170225_030002Z_cangoo_17_clean + 8 SNR mixes
wrote SSW_010_20170225_140017Z_cangoo_00_clean + 8 SNR mixes
wrote SSW_010_20170225_140017Z_cangoo_11_clean + 8 SNR mixes
wrote SSW_010_20170225_140017Z_pilwoo_14_clean + 8 SNR mixes
wrote SSW_011_20170225_160017Z_cangoo_16_clean + 8 SNR mixes
wrote SSW_015_20170225_200022Z_bkcchi_00_clean + 8 SNR mixes
wrote SSW_016_20170225_210016Z_cangoo_00_clean + 8 SNR mixes
wrote SSW_016_20170225_210016Z_cangoo_12_clean + 8 SNR mixes
wrote SSW_017_20170225_220016Z_cangoo_10_clean + 8 SNR mixes
wrote SSW_020_20170304_070004Z_cangoo_05_clean + 8 SNR mixes
wrote SSW_023_20170304_130010Z_pilwoo_01_clean + 8 SNR mixes
wrote SSW_023_20170304_130010Z_pilwoo_12_clean + 8 SNR mixes
wrote SSW_030_20170304_200023Z_cangoo_13_clean + 8 SNR mixes
wrote SSW_033_20170311_010000Z_cangoo_14_clean + 8 SNR mixes
wrote SSW_035_201